# 🌸 Task 1: Exploring and Visualizing the Iris Dataset
**DevelopersHub Corp — AI/ML Internship**

---

## 📌 Problem Statement

The goal of this task is to develop foundational data science skills by loading,
inspecting, and visualizing a real-world dataset. We use the classic **Iris dataset**
to understand:

- How to load and inspect structured data using **pandas**
- How to compute **descriptive statistics** to summarize the data
- How to create **scatter plots**, **histograms**, and **box plots** using
  **matplotlib** and **seaborn**
- How to identify **patterns**, **distributions**, and **outliers** visually

## 📂 Dataset

| Property | Value |
|---|---|
| **Name** | Iris Dataset |
| **Rows** | 150 |
| **Features** | 4 numeric + 1 categorical (species) |
| **Source** | `sklearn.datasets.load_iris()` |
| **Classes** | Setosa, Versicolor, Virginica |

---

## 1️⃣ Import Libraries

In [ ]:
# Standard data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import gaussian_kde

# Dataset source
from sklearn.datasets import load_iris

import warnings
warnings.filterwarnings('ignore')

# Consistent plot style
plt.rcParams['figure.dpi'] = 120
print('✅ Libraries imported successfully')

## 2️⃣ Load the Dataset

In [ ]:
# Load Iris from sklearn (no internet required, always available)
iris_raw = load_iris()

# Build a tidy DataFrame
df = pd.DataFrame(
    iris_raw.data,
    columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
)
df['species'] = pd.Categorical.from_codes(iris_raw.target, iris_raw.target_names)

print('✅ Dataset loaded successfully')

## 3️⃣ Data Inspection

### 3.1 Shape & Column Names

In [ ]:
print(f'Shape  : {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')

### 3.2 First 5 Rows — `.head()`

In [ ]:
df.head()

### 3.3 Dataset Info — `.info()`

> Shows data types, null counts, and memory usage.

In [ ]:
df.info()

### 3.4 Descriptive Statistics — `.describe()`

> Gives count, mean, std, min, quartiles, and max for each numeric column.

In [ ]:
df.describe().round(2)

### 3.5 Class Distribution

In [ ]:
print('Species value counts:')
print(df['species'].value_counts())
print(f'\nDataset is perfectly balanced: 50 samples per species ✅')

## 4️⃣ Data Visualization

We use a consistent color palette across all plots:
- 🩵 **Setosa** → Teal (`#4ECDC4`)
- ❤️ **Versicolor** → Coral (`#FF6B6B`)
- 💙 **Virginica** → Sky Blue (`#45B7D1`)

In [ ]:
# Global color palette
PALETTE = {'setosa': '#4ECDC4', 'versicolor': '#FF6B6B', 'virginica': '#45B7D1'}
SPECIES  = list(PALETTE.keys())
FEATURES = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
LABELS   = ['Sepal Length (cm)', 'Sepal Width (cm)', 'Petal Length (cm)', 'Petal Width (cm)']
print('Palette set ✅')

### 4.1 Scatter Plot Matrix

> Shows pairwise relationships between all four features.
> Diagonal cells show per-species histograms for that feature.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 12))
fig.suptitle('Iris Dataset — Scatter Plot Matrix', fontsize=16, fontweight='bold', y=1.01)

feat_keys = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
feat_lbls = ['Sepal Length', 'Sepal Width', 'Petal Length', 'Petal Width']

for i, (fy, ly) in enumerate(zip(feat_keys, feat_lbls)):
    for j, (fx, lx) in enumerate(zip(feat_keys, feat_lbls)):
        ax = axes[i][j]
        if i == j:  # Diagonal: histogram
            for sp, col in PALETTE.items():
                ax.hist(df[df['species'] == sp][fx], bins=14, color=col, alpha=0.7, edgecolor='none')
        else:       # Off-diagonal: scatter
            for sp, col in PALETTE.items():
                sub = df[df['species'] == sp]
                ax.scatter(sub[fx], sub[fy], color=col, alpha=0.65, s=18, edgecolors='none')
        if i == 3: ax.set_xlabel(lx, fontsize=8)
        if j == 0: ax.set_ylabel(ly, fontsize=8)
        ax.tick_params(labelsize=7)

legend_patches = [mpatches.Patch(color=c, label=s.capitalize()) for s, c in PALETTE.items()]
fig.legend(handles=legend_patches, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.02),
           fontsize=11, frameon=False)
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()

### 4.2 Histograms — Feature Distributions

> Each subplot shows how the values of one feature are distributed across the three species.
> A smooth KDE (Kernel Density Estimate) curve is overlaid for clarity.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Iris Dataset — Feature Distributions (Histograms)', fontsize=15, fontweight='bold')

for ax, feat, label in zip(axes.flat, FEATURES, LABELS):
    for sp, col in PALETTE.items():
        subset = df[df['species'] == sp][feat]
        # Histogram bars
        ax.hist(subset, bins=18, color=col, alpha=0.65, edgecolor='none', label=sp.capitalize())
        # Smooth KDE curve overlaid
        xs  = np.linspace(subset.min() - 0.3, subset.max() + 0.3, 200)
        kde = gaussian_kde(subset)
        ax.plot(xs, kde(xs) * len(subset) * (subset.max() - subset.min()) / 18,
                color=col, linewidth=2)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Value (cm)', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.legend(frameon=False, fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 4.3 Box Plots — Outlier Detection

> Box plots display the median, interquartile range (IQR), whiskers (1.5×IQR), and outliers.
> Points beyond the whiskers are flagged as potential outliers (shown as diamonds ◇).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 6))
fig.suptitle('Iris Dataset — Box Plots per Feature per Species', fontsize=15, fontweight='bold')

for ax, feat, label in zip(axes, FEATURES, LABELS):
    data_groups = [df[df['species'] == sp][feat].values for sp in SPECIES]
    bp = ax.boxplot(
        data_groups,
        patch_artist=True,
        notch=False,
        medianprops =dict(color='#222222', linewidth=2.5),
        whiskerprops=dict(linewidth=1.2),
        capprops    =dict(linewidth=1.5),
        flierprops  =dict(marker='D', markerfacecolor='gold', markeredgecolor='gray',
                          markersize=5, alpha=0.8),
    )
    for patch, col in zip(bp['boxes'], PALETTE.values()):
        patch.set_facecolor(col)
        patch.set_alpha(0.8)

    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels([s.capitalize() for s in SPECIES], fontsize=9, rotation=12)
    ax.set_title(label.replace(' (cm)', ''), fontsize=11, fontweight='bold')
    ax.set_ylabel('Value (cm)', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5️⃣ Results & Key Insights

### 📊 Summary Statistics Observations

| Feature | Mean | Std Dev | Notes |
|---|---|---|---|
| Sepal Length | 5.84 cm | 0.83 | Moderate variation across species |
| Sepal Width  | 3.06 cm | 0.44 | Least variation overall |
| Petal Length | 3.76 cm | 1.77 | **Highest variation** — great separator |
| Petal Width  | 1.20 cm | 0.76 | Also a strong species separator |

---

### 🔍 Visual Insights

**Scatter Plot Matrix:**
- **Setosa is linearly separable** from the other two species in almost every feature pair
- **Petal Length vs Petal Width** shows the clearest cluster separation
- Versicolor and Virginica overlap slightly — they'd need a non-linear boundary to separate

**Histograms:**
- Setosa has a very **narrow, low** petal distribution — easily distinguished
- Sepal Width is the only feature where **Setosa has the highest values**
- Petal Length shows a **bimodal-like** shape in the combined distribution

**Box Plots:**
- **Sepal Width** has the most outlier points (gold diamonds), especially in Setosa
- **Petal Length and Width** show almost no outliers — very clean measurements
- Virginica has the widest IQR for petal features, indicating more variability

---

### ✅ Conclusion

The Iris dataset is clean, well-balanced, and highly suitable for classification tasks.
**Petal Length** and **Petal Width** are the most discriminative features for
separating species, while **Sepal Width** is the least useful alone.
These visual insights directly inform which features to prioritize in future ML models.

---
*Task 1 Complete — DevelopersHub Corp ML Internship*